<a href="https://colab.research.google.com/github/kulka193/Squad-QA-Xform/blob/main/squadv2berttokenizedtransformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Question Answering with Transformer Model

In [ ]:
import kagglehub

### Dataset Loading (Attempted from Kaggle)

In [ ]:
squadv2_path = kagglehub.dataset_download('kulka193/squadv2')

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://www.kaggle.com/datasets/kulka193/squadv2. The server reported the following issues: Permission 'datasets.get' was denied
Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

The previous cell attempted to download the SQuAD v2 dataset from Kaggle Hub. It failed due to a 403 Client Error, indicating a permission denial. This often happens when Kaggle authentication is not set up or the dataset requires specific consent.

Therefore, we will proceed by loading the dataset from the pre-mounted `/kaggle/input` directory, which typically contains datasets provided in Kaggle environments.

In [ ]:
squadv2_path
!ls /kaggle/input/squad-v2-eval/dev-v2.0.json

/kaggle/input/squad-v2-eval/dev-v2.0.json


### Loading Evaluation Data from Local Path

In [ ]:
import json
with open('/kaggle/input/squad-v2-eval/dev-v2.0.json') as f:
    dev_eval_data = json.load(f)

### Exploring Loaded Data Structure

In [ ]:
print(dev_eval_data.keys())
for i, eval_data in enumerate(dev_eval_data['data']):
    # data
    #  L [{'title', 'paragraphs':[{'question': '...', 'answers'}],[],...},]
    print(eval_data['paragraphs'][1]['qas'])
    print(eval_data['paragraphs'][1]['context'])
    break

dict_keys(['version', 'data'])
[{'question': 'Who was the duke in the battle of Hastings?', 'id': '56dddf4066d3e219004dad5f', 'answers': [{'text': 'William the Conqueror', 'answer_start': 1022}, {'text': 'William the Conqueror', 'answer_start': 1022}, {'text': 'William the Conqueror', 'answer_start': 1022}], 'is_impossible': False}, {'question': 'Who ruled the duchy of Normandy', 'id': '56dddf4066d3e219004dad60', 'answers': [{'text': 'Richard I', 'answer_start': 573}, {'text': 'Richard I', 'answer_start': 573}, {'text': 'Richard I', 'answer_start': 573}], 'is_impossible': False}, {'question': 'What religion were the Normans', 'id': '56dddf4066d3e219004dad61', 'answers': [{'text': 'Catholic', 'answer_start': 230}, {'text': 'Catholic orthodoxy', 'answer_start': 230}, {'text': 'Catholic', 'answer_start': 230}], 'is_impossible': False}, {'plausible_answers': [{'text': 'political, cultural and military', 'answer_start': 31}], 'question': 'What type of major impact did the Norman dynasty hav

### Importing Libraries for Model Building and Data Processing

In [ ]:
from datasets import load_dataset
import torch
import json
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

### Initializing BERT Tokenizer

In [ ]:
from transformers import BertTokenizer, TFBertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

### Checking Tokenizer Class

In [ ]:
tokenizer.__class__

transformers.models.bert.tokenization_bert.BertTokenizer

### Special Tokens in Tokenizer

In [ ]:
tokenizer.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

### Tokenizer Vocabulary Size

In [ ]:
tokenizer.vocab_size

28996

### Loading SQuAD Dataset with HuggingFace `datasets`

In [ ]:
dataset = load_dataset("squad", split="train")


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

### Loading Validation Dataset

In [ ]:
eval_dataset = load_dataset("squad", split="validation")

### Displaying Dataset Information

In [ ]:
print(dataset)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})


### Data Encoding Function

The `encode` function takes a dataset example, extracts the question and the target answer based on `answer_start`, and tokenizes them using the pre-trained `BertTokenizer`. It returns the tokenized input IDs for the question and the target answer.

In [ ]:
def encode(example):
  input_text = example['question'] # f"Context: {example['context']} Question: {example['question']}"
  idx = example["answers"]["answer_start"][0]
  target_text = example["context"][idx:]
  input_encoded_text = tokenizer(input_text, add_special_tokens=True, return_tensors='pt')
  target_encoded_text = tokenizer(target_text, add_special_tokens=True, return_tensors='pt')
  #tokenizer.convert_ids_to_tokens(encoded_text['input_ids'][0])
  input_ids = input_encoded_text['input_ids'][0]
  target_ids = target_encoded_text['input_ids'][0]
  return input_ids, target_ids

### Testing the Encoding Function

In [ ]:
dataset[1]

{'id': '5733be284776f4190066117f',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'What is in front of the Notre Dame Main Building?',
 'answers': {'text': ['a copper statue of Christ'], 'answer_start': [188]}}

This cell demonstrates how the `encode` function processes a sample from the evaluation dataset, showing the tokenized question and answer. This helps verify that the tokenizer and encoding logic are working as expected.

In [ ]:
sample_q = encode(eval_dataset[1])[0]
sample_a = encode(eval_dataset[1])[1]
#print(sample_q)
#print(sample_a)
print(tokenizer.convert_ids_to_tokens(sample_q))
print(tokenizer.convert_ids_to_tokens(sample_a))

['[CLS]', 'Which', 'NFL', 'team', 'represented', 'the', 'NFC', 'at', 'Super', 'Bowl', '50', '?', '[SEP]']
['[CLS]', 'Carolina', 'Panthers', '24', '–', '10', 'to', 'earn', 'their', 'third', 'Super', 'Bowl', 'title', '.', 'The', 'game', 'was', 'played', 'on', 'February', '7', ',', '2016', ',', 'at', 'Levi', "'", 's', 'Stadium', 'in', 'the', 'San', 'Francisco', 'Bay', 'Area', 'at', 'Santa', 'Clara', ',', 'California', '.', 'As', 'this', 'was', 'the', '50th', 'Super', 'Bowl', ',', 'the', 'league', 'emphasized', 'the', '"', 'golden', 'anniversary', '"', 'with', 'various', 'gold', '-', 'themed', 'initiatives', ',', 'as', 'well', 'as', 'temporarily', 'su', '##sp', '##ending', 'the', 'tradition', 'of', 'naming', 'each', 'Super', 'Bowl', 'game', 'with', 'Roman', 'n', '##ume', '##rals', '(', 'under', 'which', 'the', 'game', 'would', 'have', 'been', 'known', 'as', '"', 'Super', 'Bowl', 'L', '"', ')', ',', 'so', 'that', 'the', 'logo', 'could', 'prominently', 'feature', 'the', 'Arabic', 'n', '##ume

### Importing `tqdm` for Progress Bars and Other Utilities

In [ ]:
from tqdm import tqdm_notebook as tqdm
import os
import time

### Custom Q&A Dataset Class

In [ ]:
class QADataset(Dataset):
  def __init__(self, dataset, max_length=128):
    self.dataset = dataset
    self.data = []
    for example in tqdm(self.dataset):
      input_ids, target_ids = encode(example)
      input_ids = input_ids[:max_length]
      #input_ids = torch.cat( input_ids, torch.zeros(max_length - len(input_ids)), dim=-1)
      #target_ids = torch.cat( target_ids, torch.zeros(max_length - len(target_ids)), dim=-1)
      tmp_ids_list = input_ids.tolist()
      tmp_ids_list = tmp_ids_list + [0] * (max_length - len(tmp_ids_list))
      input_ids = torch.tensor(tmp_ids_list)
      target_ids = target_ids[:max_length]
      if len(input_ids) <= max_length and len(target_ids) <= max_length:
        self.data.append((input_ids, target_ids))
      #if len(self.data) == 0:
      #  raise ValueError("no data points to found")
      self.max_length = max_length

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return self.data[idx]

This custom `QADataset` class inherits from `torch.utils.data.Dataset`. It preprocesses the input dataset by encoding questions and answers, and pads them to a fixed `max_length`. It stores the processed `input_ids` and `target_ids` as tensors.

In [ ]:
max_length = 64

#qsda = QADataset(dataset, max_length=max_length)
#dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

### Setting `max_length` and Initializing `batch_size`

In [ ]:
batch_size = 32
qsda.max_length

NameError: name 'qsda' is not defined

In [ ]:
qsda = QADataset(dataset, max_length=max_length)

### Checking Tokenizer's Padding Token ID

The previous cell attempted to access `qsda.max_length` before `qsda` was initialized. This section sets the `batch_size` and we'll then initialize the `QADataset` to avoid the `NameError`.

In [ ]:
tokenizer.pad_token_type_id

0

### Collate Function for DataLoader

In [ ]:
def collate_fn(batch):
  #print(batch)
  input_batch, target_batch = zip(*batch)
  input_batch = pad_sequence(input_batch, batch_first=True, padding_value=tokenizer.pad_token_type_id)
  target_batch = pad_sequence(target_batch, batch_first=True, padding_value=tokenizer.pad_token_type_id)
  return input_batch, target_batch

This `collate_fn` is essential for handling variable-length sequences in a batch. It takes a batch of `(input_ids, target_ids)` tuples, pads them to the maximum length within that batch using the tokenizer's padding token ID, and returns padded tensors for both input and target sequences.

In [ ]:
dataloader = DataLoader(qsda, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4)

### Creating DataLoader

In [ ]:
len(dataloader.dataset.data)

87599

### Importing PyTorch Modules for Model Definition

In [ ]:
import torch.nn as nn
from torch.optim import Adam

### Sequence-to-Sequence Transformer Model

In [ ]:
len(qsda[0][0])

64

In [ ]:
class Seq2SeqTransformer(nn.Module):
  def __init__(self, vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_ff, dropout=0.1):
    super(Seq2SeqTransformer, self).__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.positional_encoding = nn.Parameter(torch.zeros(1, max_length, d_model))
    self.transformer = nn.Transformer(d_model=d_model,
                                      nhead=nhead,
                                      num_encoder_layers=num_encoder_layers,
                                      num_decoder_layers=num_decoder_layers,
                                      dim_feedforward=dim_ff,
                                      dropout=dropout)
    self.fc = nn.Linear(d_model, vocab_size)
    self.dropout = nn.Dropout(dropout)
    #self.softmax = nn.Softmax(dim=1)
    #self.flatten = nn.Flatten()

  def forward(self, src, tgt):
    src_seq_len, tgt_seq_len = src.size(1), tgt.size(1)
    src = self.embedding(src)
    tgt = self.embedding(tgt)
    src += self.positional_encoding[:, :src_seq_len, :]
    tgt += self.positional_encoding[:, :tgt_seq_len, :]
    src = src.permute(1, 0, 2)
    tgt = tgt.permute(1, 0, 2)
    tgt_mask = self.transformer.generate_square_subsequent_mask(tgt_seq_len).to(tgt.device)
    output = self.transformer(src, tgt, tgt_mask=tgt_mask)
    output = self.fc(output.permute(1, 0, 2))
    #output = self.flatten(output)
    #output = self.softmax(output)
    return output

This `Seq2SeqTransformer` class defines the neural network architecture. It uses `nn.Embedding` for token embeddings, a `nn.Transformer` for the encoder-decoder mechanism, and a final linear layer (`nn.Linear`) to project the transformer output back to the vocabulary size. Positional encoding is added to capture sequence order, and a dropout layer is included for regularization.

In [ ]:
vocab_size = tokenizer.vocab_size

### Defining Vocabulary Size

In [ ]:
vocab_size

28996

### Initializing the Transformer Model

In [ ]:
model = Seq2SeqTransformer(vocab_size, 256, 4, 3, 6, 1024, dropout=0.2).to(device='cuda')

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


The `Seq2SeqTransformer` is instantiated with the defined vocabulary size and hyper-parameters (embedding dimension `d_model`, number of heads `nhead`, number of encoder/decoder layers, feedforward dimension `dim_ff`, and dropout rate). The model is then moved to the GPU (`'cuda'`) for accelerated training.

In [ ]:
model.__getattr__

<bound method Module.__getattr__ of Seq2SeqTransformer(
  (embedding): Embedding(28996, 256)
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.2, inplace=False)
          (dropout2): Dropout(p=0.2, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
       

### Setting Training Hyperparameters

In [ ]:
epochs = 100
learning_rate = 1e-4
criterion = nn.CrossEntropyLoss(ignore_index = tokenizer.pad_token_type_id)
optimizer = Adam(model.parameters(), lr=learning_rate)

Here, training parameters such as the number of epochs, learning rate, and the loss function (`CrossEntropyLoss`) are defined. The Adam optimizer is chosen for its efficiency in handling sparse gradients and adaptive learning rates.

In [ ]:
gradient_acc_steps = 4

### Importing Learning Rate Scheduler

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

### Initializing Learning Rate Scheduler

In [ ]:
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


A `ReduceLROnPlateau` scheduler is configured to monitor the validation loss (`mode='min'`). If the validation loss does not improve for `patience` number of epochs, the learning rate will be reduced by a `factor`.

In [ ]:
eval_dataset[0]

{'id': '56be4db0acb8001400a502ec',
 'title': 'Super_Bowl_50',
 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.',
 'question': 'Which NFL team represented the AFC at Super Bowl 50?',
 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],


### Sample from Evaluation Dataset

In [ ]:
def preprocess_eval_dataset(eval_data):
    return_eval_data = [] # list of tuple of (padded input_ids, padded target ids )
    count = 0
    all_input_ids = []
    all_target_ids = []
    for data_example in eval_data:
        if count == 1000:
            break
        input_ids, target_ids = encode(data_example)
        input_ids = input_ids[:max_length]
        target_ids = target_ids[:max_length]
        all_input_ids.append(input_ids)
        all_target_ids.append(target_ids)
        count += 1
    input_batch = pad_sequence(all_input_ids, batch_first=True, padding_value=tokenizer.pad_token_type_id)
    target_batch = pad_sequence(all_target_ids, batch_first=True, padding_value=tokenizer.pad_token_type_id)
    return input_batch, target_batch

### Preprocessing Evaluation Dataset

In [ ]:
eval_ds = preprocess_eval_dataset(eval_dataset)
#print(eval_ds[0], eval_ds[1])
print(eval_ds[0].shape, eval_ds[1].shape)

torch.Size([1000, 40]) torch.Size([1000, 64])


The `preprocess_eval_dataset` function processes a subset of the evaluation dataset, similar to how the training data is handled. It encodes questions and answers, pads them, and returns them as batched tensors. This is used for evaluating the model's performance during or after training.

In [ ]:
from torch.cuda.amp import autocast, GradScaler

### Mixed Precision Training Setup

In [ ]:
best_val_loss = float("inf")
patience = 3
scaler = GradScaler()
for epoch in tqdm(range(epochs)):
  model.train()
  total_loss = 0
  optimizer.zero_grad()
  for i, batch in tqdm(enumerate(dataloader)):
    src, tgt = batch
    src = src.to('cuda')
    tgt = tgt.to('cuda')
    with autocast():
        output = model(src, tgt[:, :-1])
        loss = criterion(output.reshape(-1, vocab_size), tgt[:, 1:].reshape(-1))
        loss = loss / gradient_acc_steps
    scaler.scale(loss).backward()
    if (i + 1) % gradient_acc_steps == 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
    total_loss += loss.item() * gradient_acc_steps
  #model(eval_ds[0], eval_ds[1])
  avg_loss = total_loss / len(dataloader)
  print(f"Epoch: {epoch}, Loss: {avg_loss}")

<ipython-input-50-736015c91546>:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
<ipython-input-50-736015c91546>:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for epoch in tqdm(range(epochs)):


  0%|          | 0/100 [00:00<?, ?it/s]

<ipython-input-50-736015c91546>:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i, batch in tqdm(enumerate(dataloader)):


0it [00:00, ?it/s]

<ipython-input-50-736015c91546>:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

0it [00:00, ?it/s]

Epoch: 4, Loss: 5.895061134160044


0it [00:00, ?it/s]

Epoch: 5, Loss: 5.734109685401833


0it [00:00, ?it/s]

### Training Loop

In [ ]:
torch.save(model, 'model.pth')

### Saving the Trained Model

In [ ]:
!ls /kaggle/input/squad-v2-berttransformer/pytorch/default/1

model.pth


### Loading a Pre-trained Model

In [ ]:
model = torch.load("/kaggle/input/squad-v2-berttransformer/pytorch/default/1/model.pth", map_location='cuda', weights_only=False)
#model = Seq2SeqTransformer(vocab_size, 256, 4, 3, 6, 1024, dropout=0.2).to(device='cuda')
#model.load_state_dict(state_dict)

'''
state_dict = torch.load('01.Code/models/SNNNotEncoded.pth')
model = YourModelClass()

model.load_state_dict(state_dict)
'''

"\nstate_dict = torch.load('01.Code/models/SNNNotEncoded.pth')\nmodel = YourModelClass()\n\nmodel.load_state_dict(state_dict)\n"

This cell loads a previously saved model (`model.pth`) from the `/kaggle/input` directory. The `map_location='cuda'` ensures it's loaded onto the GPU, and `weights_only=False` indicates that the entire model (architecture and state) is being loaded, not just the weights.

In [ ]:
type(eval_dataset)

datasets.arrow_dataset.Dataset

### Checking Type of Evaluation Dataset

In [ ]:
tokenizer.cls_token_id

101

### Checking CLS Token ID

In [ ]:
context = "The Eiffel Tower is a wrought-iron lattice tower"
question= "Where is Eiffel Tower located?"

### Example Context and Question for Inference

In [ ]:
model.__getattr__

<bound method Module.__getattr__ of Seq2SeqTransformer(
  (embedding): Embedding(28996, 256)
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.2, inplace=False)
          (dropout2): Dropout(p=0.2, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
       

### Greedy Answer Generation Function

In [ ]:
def generate_answer(model, context, question, max_length=50, temperature=1):
    model.eval()
    input_text = question #f"Context: {context} Question: {question}"
    encoded_text = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded_text['input_ids'][0]
    tmp_ids_list = input_ids.tolist()
    tmp_ids_list = tmp_ids_list + [0] * (max_length - len(tmp_ids_list))
    input_ids = torch.tensor([tmp_ids_list]).to('cuda')
    # Initialize target with [BOS] token
    tgt = torch.tensor([[tokenizer.cls_token_id]], device=input_ids.device)

    for _ in range(max_length):
        with torch.no_grad():
            # Forward pass: output is logits (unnormalized scores)
            output = model(input_ids, tgt)  # Shape: (batch_size, seq_len, vocab_size)

            # Get the last predicted token's logits
            next_token_logits = output[:, -1, :]  # Shape: (batch_size, vocab_size)
            scaled_logits = next_token_logits / temperature
            probs = torch.softmax(scaled_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            # Greedy decoding: select token with highest logit
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)

            # Append to the target sequence
            tgt = torch.cat([tgt, next_token], dim=1)

    # Decode the generated tokens
    answer = tokenizer.decode(tgt.squeeze().tolist(), skip_special_tokens=True)
    return answer

The `generate_answer` function implements a greedy decoding strategy for generating answers. It takes a question, tokenizes it, and then iteratively predicts the next token with the highest probability until `max_length` is reached or an end-of-sequence token is generated. `temperature` can be used to control the randomness of token selection, though in greedy decoding (argmax), its effect is limited unless directly sampling.

In [ ]:
def generate_answer_beam_search(model, context, question, max_length=50, beam_width=3):
    model.eval()
    input_text = question #f"Context: {context} Question: {question}"
    #input_ids = tokenizer.encode(input_text, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    encoded_text = tokenizer(input_text, return_tensors="pt", add_special_tokens=True).to("cuda" if torch.cuda.is_available() else "cpu")
    input_ids = encoded_text['input_ids'][0]
    tmp_ids_list = input_ids.tolist()
    tmp_ids_list = tmp_ids_list + [0] * (max_length - len(tmp_ids_list))
    input_ids = torch.tensor([tmp_ids_list]).to('cuda')
    # Initialize beam search
    beams = [{
        "tokens": torch.tensor([[tokenizer.cls_token_id]], device=input_ids.device),
        "score": 0.0,  # Log probability score
    }]

    for _ in range(max_length):
        candidates = []
        for beam in beams:
            # Stop if the beam already ends with [EOS]
            if beam["tokens"][0, -1].item() == tokenizer.eos_token_id:
                candidates.append(beam)
                continue

            # Forward pass: get logits for the next token
            with torch.no_grad():
                output = model(input_ids, beam["tokens"])
                next_token_logits = output[:, -1, :]  # Shape: (1, vocab_size)
                next_token_probs = torch.log_softmax(next_token_logits, dim=-1)  # Log probabilities

            # Top-k candidates for this beam
            topk_probs, topk_tokens = torch.topk(next_token_probs, beam_width)
            topk_probs = topk_probs.squeeze().tolist()
            topk_tokens = topk_tokens.squeeze().tolist()

            # Create new candidates
            for prob, token in zip(topk_probs, topk_tokens):
                new_tokens = torch.cat([beam["tokens"], torch.tensor([[token]], device=input_ids.device)], dim=1)
                #print(new_tokens)
                new_score = beam["score"] + prob
                candidates.append({
                    "tokens": new_tokens,
                    "score": new_score,
                })

        # Select top-k candidates overall
        candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)[:beam_width]
        beams = candidates

    # Select the beam with the highest score
    best_beam = beams[0]
    answer = tokenizer.decode(best_beam["tokens"].squeeze().tolist(), skip_special_tokens=True)
    return answer

### Beam Search Answer Generation Function

In [ ]:
import random
class ShuffleDataset(torch.utils.data.IterableDataset):
  def __init__(self, dataset, buffer_size):
    super().__init__()
    self.dataset = dataset
    self.buffer_size = buffer_size

  def __iter__(self):
    shufbuf = []
    try:
      dataset_iter = iter(self.dataset)
      for i in range(self.buffer_size):
        shufbuf.append(next(dataset_iter))
    except:
      self.buffer_size = len(shufbuf)

    try:
      while True:
        try:
          item = next(dataset_iter)
          evict_idx = random.randint(0, self.buffer_size - 1)
          yield shufbuf[evict_idx]
          shufbuf[evict_idx] = item
        except StopIteration:
          break
      while len(shufbuf) > 0:
        yield shufbuf.pop()
    except GeneratorExit:
      pass

### Custom Shuffle Dataset Class

In [ ]:
shuffled_eval_dataset = ShuffleDataset(eval_dataset, 1024)

The `ShuffleDataset` class is a custom `IterableDataset` that shuffles the input dataset using a buffer. This is particularly useful for large datasets that cannot fit entirely into memory, providing a way to introduce randomness during iteration without loading the entire dataset.

In [ ]:
for ds in shuffled_eval_dataset:
    print(ds)
    break

{'id': '56beace93aeaaa14008c91e0', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'What venue did Super Bowl 50 take place in?', 'answers': {'text': ["Levi's Stadium", "Levi's Stadium", "Levi's Stadium in the San Franc

### Testing Shuffled Evaluation Dataset

In [ ]:
context = "Where is Taj Mahal"
question= "What is in front of the Notre Dame Main Building?"
num_inferences = 1000
i = 0
for ed in shuffled_eval_dataset:
    response =generate_answer_beam_search(model, context="", question=ed['question'], max_length=50, beam_width=3)
    print(f'Question: {ed["question"]} \n Reponse: {response}')
    if i == num_inferences:
        break
    i += 1
    print('\n \n')

Question: Who designed the Vince Lombardi Trophy? 
 Reponse: Antonio La Ferreira, a popular example of the Bois de Vincennese. The most famous façades are the Charminar, the Mecca Mauricio de Bellas, the Basilica of Valais, the Basilica of Valiant,

 

Question: When was the last time California hosted a Super Bowl? 
 Reponse: 1927. The Washington University of Washington Medical Center is home to Charlotte's first major professional sports team in Washington, Washington, D. C. C., in Washington, D. C., Washington, D. C., and Washington D

 

Question: Which video gaming company debuted their ad for the first time during Super Bowl 50? 
 Reponse: Pocket - based video game console in the U. S. - based videoconferencing became the first commercial videoconferencing console in the United States.s of the indie - based videoconferencing systems in the United States, and

 

Question: When were the two finalists for hosting Super Bowl 50 announced? 
 Reponse: January 7, 2006, 2010, the Ameri

### Running Inference with Beam Search

In [ ]:
!cp -r /kaggle/working/model.pth /kaggle/input

### Copying Model to Kaggle Input

In [ ]:
!ls /kaggle/input/squadv2